# Training

## Loading real mawi trace dataset from csv

In [5]:
import pandas as pd
from pathlib import Path

benign_csv_path = Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/mawi20260000_first3_features.csv")

if not benign_csv_path.exists():
    raise FileNotFoundError(f"File not found: {benign_csv_path}")

benign_df = pd.read_csv(benign_csv_path)

print("Loaded:", benign_csv_path)
print("Rows:", benign_df.shape[0])
print("Columns:", benign_df.shape[1])

print("Column names:")
print(list(benign_df.columns))

display(benign_df.head())

Loaded: /home/ubuntu/DDoS_ML/preprocessing/features_out/mawi20260000_first3_features.csv
Rows: 2476811
Columns: 25
Column names:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'packet_count', 'duration', 'total_bytes', 'packets_per_second', 'bytes_per_second', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'iat_min', 'iat_max', 'iat_mean', 'iat_std', 'tcp_syn_count', 'tcp_ack_count', 'tcp_fin_count', 'tcp_rst_count', 'tcp_psh_count', 'tcp_urg_count', 'label']


,src_ip,dst_ip,src_port,dst_port,protocol,packet_count,duration,total_bytes,packets_per_second,bytes_per_second,...,iat_max,iat_mean,iat_std,tcp_syn_count,tcp_ack_count,tcp_fin_count,tcp_rst_count,tcp_psh_count,tcp_urg_count,label
0,157.209.43.67,47.173.135.177,80,48978,TCP,3,0.000066,198,45425.675090,2.998095e+06,...,0.000035,0.000033,0.000003,0,3,0,0,0,0,Benign
1,146.195.151.206,150.189.32.126,443,32840,TCP,3,0.000031,198,96791.630769,6.388248e+06,...,0.000022,0.000015,0.000009,0,3,0,0,1,0,Benign
2,2001:4dfb:ea23:2c79:2d7c:7441:5e5c:1033,2404:643b:cb88:300e:7eff:8b8c:27f0:7f41,57765,443,UDP,3,0.000078,186,38479.853211,2.385751e+06,...,0.000076,0.000039,0.000052,0,0,0,0,0,0,Benign
3,71.81.13.209,150.189.109.193,443,58208,TCP,3,0.000010,198,299593.142857,1.977315e+07,...,0.000006,0.000005,0.000002,0,3,0,0,0,0,Benign
4,150.189.111.84,162.103.113.232,61885,443,UDP,3,0.000004,126,786432.000000,3.303014e+07,...,0.000003,0.000002,0.000001,0,0,0,0,0,0,Benign


## Loading DDoS data from multiple datasets

### Loading slow attacks from DDoD-AT-2022 and other attacks from CICDDoS2019

In [10]:
import pandas as pd
from pathlib import Path

# List attack CSV files here
attack_csv_paths = [
    Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/HTTP-slow-header_first3_features.csv"),
    Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/TCP-SYN-low_first3_features.csv"),
    Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/03-11/03-11/batch/combined_features_n3.csv"),
]

# Columns I want to remove before joining
columns_to_drop = [
    "pcap_file",
    "relative_path",
    "traffic_type",
    "label_source",
    "csv_match_time_diff",
    "flow_start_time",
    "flow_end_time"
]

# Labels to force for specific CSV files
label_overrides = {
    Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/HTTP-slow-header_first3_features.csv"): "HTTP-slow",
    Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/TCP-SYN-low_first3_features.csv"): "TCP-SYN-low",
}

### Dropping unwanted columns and over-writing labels

In [12]:
attack_dfs = []

for csv_path in attack_csv_paths:
    print("=" * 80)
    print("Loading:", csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"File not found: {csv_path}")

    temp_df = pd.read_csv(csv_path)

    print("Shape before dropping columns:", temp_df.shape)
    print("Columns:")
    print(list(temp_df.columns))

    existing_drop_cols = [col for col in columns_to_drop if col in temp_df.columns]

    if existing_drop_cols:
        temp_df = temp_df.drop(columns=existing_drop_cols)
        print("Dropped columns:", existing_drop_cols)
    else:
        print("No columns dropped.")

    if csv_path in label_overrides:
        temp_df["label"] = label_overrides[csv_path]
        print("Overwrote label with:", label_overrides[csv_path])
    else:
        print("Kept original label column.")

    print("Shape after dropping columns:", temp_df.shape)

    attack_dfs.append({
        "path": csv_path,
        "df": temp_df
    })

Loading: /home/ubuntu/DDoS_ML/preprocessing/features_out/HTTP-slow-header_first3_features.csv
Shape before dropping columns: (18116, 28)
Columns:
['pcap_file', 'relative_path', 'traffic_type', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'packet_count', 'duration', 'total_bytes', 'packets_per_second', 'bytes_per_second', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'iat_min', 'iat_max', 'iat_mean', 'iat_std', 'tcp_syn_count', 'tcp_ack_count', 'tcp_fin_count', 'tcp_rst_count', 'tcp_psh_count', 'tcp_urg_count', 'label']
Dropped columns: ['pcap_file', 'relative_path', 'traffic_type']
Overwrote label with: HTTP-slow
Shape after dropping columns: (18116, 25)
Loading: /home/ubuntu/DDoS_ML/preprocessing/features_out/TCP-SYN-low_first3_features.csv
Shape before dropping columns: (64016, 28)
Columns:
['pcap_file', 'relative_path', 'traffic_type', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'packet_count', 'duration', 'total_bytes', 

### Checking for exact match for columns in attack and benign data

In [13]:
# =========================
# Check attack dataframe columns
# =========================

reference_attack_path = attack_dfs[0]["path"]
reference_attack_cols = list(attack_dfs[0]["df"].columns)

attack_columns_match = True

for item in attack_dfs:
    current_path = item["path"]
    current_cols = list(item["df"].columns)

    print("=" * 80)
    print("Checking:", current_path)

    if current_cols == reference_attack_cols:
        print("Columns match exactly and are in the same order.")
    else:
        attack_columns_match = False
        print("Columns DO NOT match exactly.")

        missing_from_current = [col for col in reference_attack_cols if col not in current_cols]
        extra_in_current = [col for col in current_cols if col not in reference_attack_cols]

        print("Missing columns compared to reference:")
        print(missing_from_current)

        print("Extra columns compared to reference:")
        print(extra_in_current)

        print("Column order / position differences:")
        max_len = max(len(reference_attack_cols), len(current_cols))

        for i in range(max_len):
            ref_col = reference_attack_cols[i] if i < len(reference_attack_cols) else None
            cur_col = current_cols[i] if i < len(current_cols) else None

            if ref_col != cur_col:
                print(f"Position {i}: reference={ref_col}, current={cur_col}")

Checking: /home/ubuntu/DDoS_ML/preprocessing/features_out/HTTP-slow-header_first3_features.csv
Columns match exactly and are in the same order.
Checking: /home/ubuntu/DDoS_ML/preprocessing/features_out/TCP-SYN-low_first3_features.csv
Columns match exactly and are in the same order.
Checking: /home/ubuntu/DDoS_ML/preprocessing/features_out/03-11/03-11/batch/combined_features_n3.csv
Columns match exactly and are in the same order.


In [14]:
# =========================
# Check attack columns vs benign columns
# =========================

benign_cols = list(benign_df.columns)
attack_cols = reference_attack_cols

print("=" * 80)
print("Checking attack columns against benign_df columns")

if attack_cols == benign_cols:
    print("Attack and benign columns match exactly and are in the same order.")
else:
    print("Attack and benign columns DO NOT match exactly.")

    missing_from_attack = [col for col in benign_cols if col not in attack_cols]
    extra_in_attack = [col for col in attack_cols if col not in benign_cols]

    print("Columns in benign_df but missing from attack:")
    print(missing_from_attack)

    print("Columns in attack but missing from benign_df:")
    print(extra_in_attack)

    print("Column order / position differences:")
    max_len = max(len(benign_cols), len(attack_cols))

    for i in range(max_len):
        benign_col = benign_cols[i] if i < len(benign_cols) else None
        attack_col = attack_cols[i] if i < len(attack_cols) else None

        if benign_col != attack_col:
            print(f"Position {i}: benign={benign_col}, attack={attack_col}")

    raise ValueError("Attack and benign columns do not match exactly. Fix this before training.")

Checking attack columns against benign_df columns
Attack and benign columns match exactly and are in the same order.


### Joining all dataframes in attack_df

In [15]:
attack_df = pd.concat(
    [item["df"] for item in attack_dfs],
    ignore_index=True
)

print("Final attack_df shape:", attack_df.shape)
print("Labels in final attack_df:")
print(attack_df["label"].value_counts())

display(attack_df.head())

Final attack_df shape: (2851717, 25)
Labels in final attack_df:
label
UDP            1941505
Syn             754745
TCP-SYN-low      64016
BENIGN           23744
unknown          20747
MSSQL            19417
HTTP-slow        18116
NetBIOS           6628
LDAP              2286
Portmap            363
UDPLag             150
Name: count, dtype: int64


,src_ip,dst_ip,src_port,dst_port,protocol,packet_count,duration,total_bytes,packets_per_second,bytes_per_second,...,iat_max,iat_mean,iat_std,tcp_syn_count,tcp_ack_count,tcp_fin_count,tcp_rst_count,tcp_psh_count,tcp_urg_count,label
0,10.10.0.10,192.168.10.11,36400,80,TCP,3,0.001158,218,2590.675726,188255.769405,...,0.001157,0.000579,0.000817,2,1,0,0,0,0,HTTP-slow
1,10.10.0.10,192.168.10.11,36398,80,TCP,3,0.001153,218,2601.925558,189073.257237,...,0.001153,0.000576,0.000815,2,1,0,0,0,0,HTTP-slow
2,192.168.10.11,10.10.0.10,80,36398,TCP,3,0.005042,218,594.993002,43236.158124,...,0.005042,0.002521,0.003565,2,3,0,0,0,0,HTTP-slow
3,192.168.10.11,10.10.0.10,80,36400,TCP,3,0.005042,218,595.021138,43238.202677,...,0.005042,0.002521,0.003565,2,3,0,0,0,0,HTTP-slow
4,10.10.0.10,192.168.10.11,36402,80,TCP,3,0.000799,218,3753.852029,272779.914081,...,0.000799,0.000400,0.000565,2,1,0,0,0,0,HTTP-slow


### Dropping sample based on labels from the attach data

In [16]:
# =========================
# Drop attack_df rows by label
# =========================

label_column = "label"

if label_column not in attack_df.columns:
    raise KeyError(f"Column '{label_column}' not found. Available columns are: {list(attack_df.columns)}")

print("Labels before dropping:")
print(attack_df[label_column].value_counts())

labels_to_drop = [
    "BENIGN",
    "unknown",
    # add labels you want to remove here
]

attack_df = attack_df[~attack_df[label_column].isin(labels_to_drop)].copy()

print("\nLabels after dropping:")
print(attack_df[label_column].value_counts())

print("\nNew attack_df shape:", attack_df.shape)

Labels before dropping:
label
UDP            1941505
Syn             754745
TCP-SYN-low      64016
BENIGN           23744
unknown          20747
MSSQL            19417
HTTP-slow        18116
NetBIOS           6628
LDAP              2286
Portmap            363
UDPLag             150
Name: count, dtype: int64

Labels after dropping:
label
UDP            1941505
Syn             754745
TCP-SYN-low      64016
MSSQL            19417
HTTP-slow        18116
NetBIOS           6628
LDAP              2286
Portmap            363
UDPLag             150
Name: count, dtype: int64

New attack_df shape: (2807226, 25)


### Creating ddos_df for binary classification

In [17]:
# =========================
# Create binary DDoS dataframe
# =========================

ddos_df = attack_df.copy()

ddos_df["label"] = "ddos"

print("ddos_df shape:", ddos_df.shape)

print("Labels in ddos_df:")
print(ddos_df["label"].value_counts())

display(ddos_df.head())

ddos_df shape: (2807226, 25)
Labels in ddos_df:
label
ddos    2807226
Name: count, dtype: int64


,src_ip,dst_ip,src_port,dst_port,protocol,packet_count,duration,total_bytes,packets_per_second,bytes_per_second,...,iat_max,iat_mean,iat_std,tcp_syn_count,tcp_ack_count,tcp_fin_count,tcp_rst_count,tcp_psh_count,tcp_urg_count,label
0,10.10.0.10,192.168.10.11,36400,80,TCP,3,0.001158,218,2590.675726,188255.769405,...,0.001157,0.000579,0.000817,2,1,0,0,0,0,ddos
1,10.10.0.10,192.168.10.11,36398,80,TCP,3,0.001153,218,2601.925558,189073.257237,...,0.001153,0.000576,0.000815,2,1,0,0,0,0,ddos
2,192.168.10.11,10.10.0.10,80,36398,TCP,3,0.005042,218,594.993002,43236.158124,...,0.005042,0.002521,0.003565,2,3,0,0,0,0,ddos
3,192.168.10.11,10.10.0.10,80,36400,TCP,3,0.005042,218,595.021138,43238.202677,...,0.005042,0.002521,0.003565,2,3,0,0,0,0,ddos
4,10.10.0.10,192.168.10.11,36402,80,TCP,3,0.000799,218,3753.852029,272779.914081,...,0.000799,0.000400,0.000565,2,1,0,0,0,0,ddos


## Feature selection

In [18]:
# =========================
# Prepare benign labels
# =========================

benign_df = benign_df.copy()
benign_df["label"] = "benign"

print("benign_df shape:", benign_df.shape)
print(benign_df["label"].value_counts())

benign_df shape: (2476811, 25)
label
benign    2476811
Name: count, dtype: int64


In [19]:
# =========================
# Combine benign and DDoS data
# =========================

model_df = pd.concat([benign_df, ddos_df], ignore_index=True)

print("model_df shape:", model_df.shape)

print("Labels:")
print(model_df["label"].value_counts())

display(model_df.head())

model_df shape: (5284037, 25)
Labels:
label
ddos      2807226
benign    2476811
Name: count, dtype: int64


,src_ip,dst_ip,src_port,dst_port,protocol,packet_count,duration,total_bytes,packets_per_second,bytes_per_second,...,iat_max,iat_mean,iat_std,tcp_syn_count,tcp_ack_count,tcp_fin_count,tcp_rst_count,tcp_psh_count,tcp_urg_count,label
0,157.209.43.67,47.173.135.177,80,48978,TCP,3,0.000066,198,45425.675090,2.998095e+06,...,0.000035,0.000033,0.000003,0,3,0,0,0,0,benign
1,146.195.151.206,150.189.32.126,443,32840,TCP,3,0.000031,198,96791.630769,6.388248e+06,...,0.000022,0.000015,0.000009,0,3,0,0,1,0,benign
2,2001:4dfb:ea23:2c79:2d7c:7441:5e5c:1033,2404:643b:cb88:300e:7eff:8b8c:27f0:7f41,57765,443,UDP,3,0.000078,186,38479.853211,2.385751e+06,...,0.000076,0.000039,0.000052,0,0,0,0,0,0,benign
3,71.81.13.209,150.189.109.193,443,58208,TCP,3,0.000010,198,299593.142857,1.977315e+07,...,0.000006,0.000005,0.000002,0,3,0,0,0,0,benign
4,150.189.111.84,162.103.113.232,61885,443,UDP,3,0.000004,126,786432.000000,3.303014e+07,...,0.000003,0.000002,0.000001,0,0,0,0,0,0,benign


In [20]:
# =========================
# Separate X and y
# =========================

target_column = "label"

X = model_df.drop(columns=[target_column])
y = model_df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Feature columns:")
print(list(X.columns))

X shape: (5284037, 24)
y shape: (5284037,)
Feature columns:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'packet_count', 'duration', 'total_bytes', 'packets_per_second', 'bytes_per_second', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'iat_min', 'iat_max', 'iat_mean', 'iat_std', 'tcp_syn_count', 'tcp_ack_count', 'tcp_fin_count', 'tcp_rst_count', 'tcp_psh_count', 'tcp_urg_count']


In [21]:
# =========================
# Check feature data types
# =========================

print(X.dtypes)

src_ip                 object
dst_ip                 object
src_port                int64
dst_port                int64
protocol               object
packet_count            int64
duration              float64
total_bytes             int64
packets_per_second    float64
bytes_per_second      float64
packet_length_min       int64
packet_length_max       int64
packet_length_mean    float64
packet_length_std     float64
iat_min               float64
iat_max               float64
iat_mean              float64
iat_std               float64
tcp_syn_count           int64
tcp_ack_count           int64
tcp_fin_count           int64
tcp_rst_count           int64
tcp_psh_count           int64
tcp_urg_count           int64
dtype: object


In [22]:
# List non-numeric feature columns
non_numeric_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Non-numeric columns:")
print(non_numeric_cols)

Non-numeric columns:
['src_ip', 'dst_ip', 'protocol']


In [23]:
columns_to_remove_from_features = [
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "protocol",
]

X = X.drop(columns=[col for col in columns_to_remove_from_features if col in X.columns])

print("X shape after removing non-feature columns:", X.shape)
print("Remaining feature columns:")
print(list(X.columns))

X shape after removing non-feature columns: (5284037, 19)
Remaining feature columns:
['packet_count', 'duration', 'total_bytes', 'packets_per_second', 'bytes_per_second', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'iat_min', 'iat_max', 'iat_mean', 'iat_std', 'tcp_syn_count', 'tcp_ack_count', 'tcp_fin_count', 'tcp_rst_count', 'tcp_psh_count', 'tcp_urg_count']


In [25]:
# =========================
# Keep only numeric features
# =========================

non_numeric_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Non-numeric columns that will be removed:")
print(non_numeric_cols)

X_numeric = X.drop(columns=non_numeric_cols)

print("Original X shape:", X.shape)
print("Numeric X shape:", X_numeric.shape)

Non-numeric columns that will be removed:
[]
Original X shape: (5284037, 19)
Numeric X shape: (5284037, 19)


In [26]:
# =========================
# Check missing and infinite values
# =========================

import numpy as np

print("Total missing values:")
print(X_numeric.isna().sum().sum())

print("Total infinite values:")
print(np.isinf(X_numeric).sum().sum())

Total missing values:
0
Total infinite values:
0


In [27]:
# =========================
# Check labels
# =========================

print("Labels:")
print(y.value_counts())

Labels:
label
ddos      2807226
benign    2476811
Name: count, dtype: int64


In [29]:
# =========================
# Train/test split
# =========================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

X_train shape: (4227229, 19)
X_test shape: (1056808, 19)

Training labels:
label
ddos      2245780
benign    1981449
Name: count, dtype: int64

Testing labels:
label
ddos      561446
benign    495362
Name: count, dtype: int64


In [30]:
# =========================
# Train initial Random Forest
# =========================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [31]:
# =========================
# Feature importance
# =========================

feature_importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance_df)

,feature,importance
0,iat_min,0.339491
1,packet_length_max,0.156123
2,total_bytes,0.145857
3,packet_length_mean,0.098344
4,packet_length_min,0.075065
5,bytes_per_second,0.031101
6,iat_std,0.028461
7,tcp_syn_count,0.028452
8,packet_length_std,0.021798
9,iat_max,0.019547


In [32]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9999886450518921

Confusion matrix:
[[495354      8]
 [     4 561442]]

Classification report:
              precision    recall  f1-score   support

      benign       1.00      1.00      1.00    495362
        ddos       1.00      1.00      1.00    561446

    accuracy                           1.00   1056808
   macro avg       1.00      1.00      1.00   1056808
weighted avg       1.00      1.00      1.00   1056808

